In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

# Load the dataset
df = pd.read_csv("Task 3 and 4_Loan_Data.csv")

# Take a quick look at the data
df.head()

,customer_id,credit_lines_outstanding,loan_amt_outstanding,total_debt_outstanding,income,years_employed,fico_score,default
0,8153374,0,5221.545193,3915.471226,78039.38546,5,605,0
1,7442532,5,1958.928726,8228.752520,26648.43525,2,572,1
2,2256073,0,3363.009259,2027.830850,65866.71246,4,602,0
3,4885975,0,4766.648001,2501.730397,74356.88347,5,612,0
4,4700614,1,1345.827718,1768.826187,23448.32631,6,631,0


In [2]:
# Separate features (X) and target (y)
# We drop 'customer_id' because random ID numbers do not predict risk
X = df.drop(columns=['customer_id', 'default'])
y = df['default']

# Split the data into a training set (80%) and a testing set (20%)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Scale the data using StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)

# We scale the test set based on the training set's metrics
X_test_scaled = scaler.transform(X_test)

In [3]:
# Initialize and train the Logistic Regression model
model = LogisticRegression()
model.fit(X_train_scaled, y_train)

# (Optional) Check the accuracy on our test set
accuracy = model.score(X_test_scaled, y_test)
print(f"Model Accuracy on Test Data: {accuracy * 100:.2f}%")

Model Accuracy on Test Data: 99.55%


In [4]:
def calculate_expected_loss(credit_lines, loan_amt, total_debt, income, years_employed, fico_score):
    """
    Takes borrower properties and returns the Expected Loss in dollars.
    Assumes a 10% recovery rate.
    """
    
    # 1. Format the new borrower's data exactly like our original dataframe
    borrower_data = pd.DataFrame([[credit_lines, loan_amt, total_debt, income, years_employed, fico_score]], 
                                 columns=['credit_lines_outstanding', 'loan_amt_outstanding', 'total_debt_outstanding', 
                                          'income', 'years_employed', 'fico_score'])
    
    # 2. Scale the inputs using the SAME scaler we fitted earlier
    borrower_scaled = scaler.transform(borrower_data)
    
    # 3. Predict the Probability of Default (PD)
    # .predict_proba() returns a list with two probabilities: [Prob of 0, Prob of 1]
    # We want the probability of 1 (Default), which is at index [0][1]
    pd_value = model.predict_proba(borrower_scaled)[0][1]
    
    # 4. Calculate Expected Loss (EL = EAD * PD * LGD)
    recovery_rate = 0.10
    loss_given_default = 1 - recovery_rate
    
    expected_loss = loan_amt * pd_value * loss_given_default
    
    return expected_loss, pd_value

In [5]:
# Let's test two theoretical borrowers

# Borrower A: Good credit, high income, low debt
el_A, pd_A = calculate_expected_loss(
    credit_lines=1, 
    loan_amt=5000, 
    total_debt=1500, 
    income=85000, 
    years_employed=5, 
    fico_score=710
)

# Borrower B: Bad credit, low income, high debt
el_B, pd_B = calculate_expected_loss(
    credit_lines=5, 
    loan_amt=5000, 
    total_debt=8000, 
    income=30000, 
    years_employed=1, 
    fico_score=550
)

print(f"Borrower A - Probability of Default: {pd_A*100:.2f}% | Expected Loss: ${el_A:.2f}")
print(f"Borrower B - Probability of Default: {pd_B*100:.2f}% | Expected Loss: ${el_B:.2f}")

Borrower A - Probability of Default: 0.00% | Expected Loss: $0.00
Borrower B - Probability of Default: 100.00% | Expected Loss: $4500.00
